In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.cluster.hierarchy as sch
import os
import warnings

# Silenciar avisos para uma execução limpa
warnings.filterwarnings('ignore')

def mapear_correlacao_e_diversificacao(diretorio_dados, top_n_pares=50):
    print("=== INÍCIO DO ALGORITMO DE CORRELAÇÃO E CLUSTERIZAÇÃO HIERÁRQUICA ===")
    
    # 1. Carregamento da Base de Dados
    caminho_entrada = os.path.join(diretorio_dados, "matriz_premio_risco.csv")
    print("1. A carregar vetores de retornos em excesso...")
    df_er = pd.read_csv(caminho_entrada, index_col='Data', parse_dates=True)
    
    # Isolar apenas as ações (remover o IBOV para não poluir a matriz cruzada)
    colunas_ativos = [col for col in df_er.columns if col != 'Premio_Mercado_IBOV']
    df_ativos = df_er[colunas_ativos]
    
    # 2. Cálculo da Matriz de Correlação de Pearson
    print("2. A processar a Matriz de Correlação de Pearson global...")
    matriz_correlacao = df_ativos.corr(method='pearson')
    
    # 3. Renderização do Heatmap Clusterizado (Machine Learning Não-Supervisionado)
    print("3. A desenhar o Clustermap (Mapa de Calor com Agrupamento Hierárquico)...")
    
    # Configuração de cores: 'coolwarm' vai do azul (descorrelacionado/negativo) ao vermelho (altamente correlacionado)
    cmap_personalizado = sns.color_palette("coolwarm", as_cmap=True)
    
    # O Clustermap reorganiza as colunas e linhas para juntar os ativos parecidos
    g = sns.clustermap(matriz_correlacao, 
                       method='ward',      # Método de variância mínima para formar os clusters
                       metric='euclidean', # Medida de distância entre as ações
                       cmap=cmap_personalizado, 
                       figsize=(22, 22),   # Tamanho gigante para acomodar 136 ações
                       linewidths=0.0, 
                       vmin=-0.2, vmax=1.0,# Forçar a escala de cores para realçar pequenas diferenças
                       cbar_kws={'label': 'Coeficiente de Correlação (Pearson)', 'shrink': 0.5})
    
    # Ajustes estéticos para o gráfico da tese
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=90, fontsize=6)
    plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0, fontsize=6)
    g.fig.suptitle('Matriz de Correlação Hierárquica: Agrupamento Estrutural e Diversificação', y=1.02, fontsize=20)
    
    # Guardar a imagem em altíssima resolução
    caminho_grafico = os.path.join(diretorio_dados, "heatmap_correlacao_clusterizada.png")
    g.savefig(caminho_grafico, dpi=400, bbox_inches='tight')
    plt.close()
    
    # 4. Extração Cirúrgica dos Pares Descorrelacionados
    print("4. A extrair o ranking de máxima diversificação (menor correlação)...")
    
    # Extrair apenas o triângulo inferior para não duplicar pares (ex: VALE-ITUB e ITUB-VALE)
    matriz_lower = matriz_correlacao.where(np.tril(np.ones(matriz_correlacao.shape), k=-1).astype(bool))
    
    # Empilhar (transformar de matriz para tabela) e limpar
    pares = matriz_lower.stack().reset_index()
    pares.columns = ['Ativo_1', 'Ativo_2', 'Correlacao']
    
    # Ordenar pelos valores MAIS BAIXOS de correlação
    # Estes são os ativos que "menos ligam" para o que o outro faz
    pares_descorrelacionados = pares.sort_values(by='Correlacao', ascending=True).head(top_n_pares)
    
    # 5. Exportação da Tabela
    caminho_pares = os.path.join(diretorio_dados, "ranking_pares_descorrelacionados.csv")
    pares_descorrelacionados.to_csv(caminho_pares, index=False)
    
    print("\n=== RESUMO DA ANÁLISE DE DIVERSIFICAÇÃO ===")
    print(f"Total de ações cruzadas: {len(colunas_ativos)}")
    print(f"Correlação média do mercado brasileiro na amostra: {matriz_lower.mean().mean():.4f}")
    print(f"Menor correlação encontrada: {pares_descorrelacionados.iloc[0]['Correlacao']:.4f} (Par: {pares_descorrelacionados.iloc[0]['Ativo_1']} / {pares_descorrelacionados.iloc[0]['Ativo_2']})")
    print(f"Heatmap Clusterizado salvo em: {caminho_grafico}")
    print(f"Ranking dos {top_n_pares} Pares de Ouro salvo em: {caminho_pares}")
    print("=========================================================================")
    
    return pares_descorrelacionados

# ==========================================
# ÁREA DE EXECUÇÃO
# ==========================================
pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"

# Executa e visualiza os top 15 pares na tela
ranking_diversificacao = mapear_correlacao_e_diversificacao(pasta_dados, top_n_pares=100)

print("\n🏆 TOP 15 PARES PARA MÁXIMA DIVERSIFICAÇÃO NO BRASIL:")
print(ranking_diversificacao.head(15).to_string(index=False))

=== INÍCIO DO ALGORITMO DE CORRELAÇÃO E CLUSTERIZAÇÃO HIERÁRQUICA ===
1. A carregar vetores de retornos em excesso...
2. A processar a Matriz de Correlação de Pearson global...
3. A desenhar o Clustermap (Mapa de Calor com Agrupamento Hierárquico)...
4. A extrair o ranking de máxima diversificação (menor correlação)...

=== RESUMO DA ANÁLISE DE DIVERSIFICAÇÃO ===
Total de ações cruzadas: 136
Correlação média do mercado brasileiro na amostra: 0.2058
Menor correlação encontrada: -0.0025 (Par: RCSL4 / GOLL54)
Heatmap Clusterizado salvo em: C:\VSCodeWorkspace\TCC_Escrito\data\heatmap_correlacao_clusterizada.png
Ranking dos 100 Pares de Ouro salvo em: C:\VSCodeWorkspace\TCC_Escrito\data\ranking_pares_descorrelacionados.csv

🏆 TOP 15 PARES PARA MÁXIMA DIVERSIFICAÇÃO NO BRASIL:
Ativo_1 Ativo_2  Correlacao
  RCSL4  GOLL54   -0.002513
  LUPA3   FICT3    0.004662
  ISAE4   FICT3    0.005182
  TELB4   FICT3    0.007572
  SLCE3  GOLL54    0.008099
  PDGR3  GOLL54    0.008256
  SANB4   LUPA3    0.0